# 02 长期记忆 — 跨会话信息持久化

01 框架的 `ConversationMemory` 是短期记忆——程序退出就丢失。
长期记忆让 Agent 能跨会话记住重要信息。

**学习点**：JSON 持久化、关键词检索、自动记忆提取、上下文注入

In [ ]:
import sys, json, os, shutil
sys.path.insert(0, '..')

from src.agent_framework import Agent, LongTermMemory

# 清理旧测试数据
if os.path.exists('agent_memory'):
    shutil.rmtree('agent_memory')

print('一切就绪')

## LongTermMemory 基础用法

不依赖 Agent，可以独立使用。

In [ ]:
ltm = LongTermMemory()

# 存储记忆
ltm.add('用户叫程宣赫，在人民大学读书')
ltm.add('用户喜欢 Python 编程，关注 AI 方向')
ltm.add('用户下个月要去东京旅游')

print('所有记忆:')
for m in ltm.list_all():
    print(f"  [{m['index']}] {m['content']}  ({m['timestamp']})")

# 检索
print('\n搜索 "北京 学校":')
for r in ltm.search('北京 学校'):
    print(f"  - {r['content']}")

print('\n搜索 "Python AI":')
for r in ltm.search('Python AI'):
    print(f"  - {r['content']}")

## 持久化验证

创建新实例，数据不会丢。

In [ ]:
# 删除旧实例，创建新的
del ltm
ltm2 = LongTermMemory()
print(f'新实例读取到 {len(ltm2.list_all())} 条记忆')
for m in ltm2.list_all():
    print(f"  [{m['index']}] {m['content']}")

# 清理
shutil.rmtree('agent_memory')
print('\n持久化验证通过')

## Agent + 长期记忆集成

给 Agent 注入长期记忆，检索 + 上下文注入全自动。

In [ ]:
# 准备工具
def get_weather(city):
    w = {'北京': {'temp': 25, 'desc': '晴朗'}, '上海': {'temp': 28, 'desc': '多云'},
         '东京': {'temp': 22, 'desc': '小雨'}}
    for k, v in w.items():
        if k in city:
            return json.dumps(v, ensure_ascii=False)
    return json.dumps({'temp': 20, 'desc': '未知'}, ensure_ascii=False)

def save_memory(content):
    ltm3.add(content)
    return '记忆已保存'

DEMO_TOOLS = [
    {'name': 'get_weather', 'description': '查询指定城市的天气',
     'parameters': {'type': 'object', 'properties': {'city': {'type': 'string'}}, 'required': ['city']},
     'fn': get_weather},
    {'name': 'save_memory', 'description': '保存重要信息到长期记忆，当用户告诉你关于自己的事实或偏好时调用',
     'parameters': {'type': 'object', 'properties': {'content': {'type': 'string', 'description': '要记住的内容'}}, 'required': ['content']},
     'fn': save_memory},
]

ltm3 = LongTermMemory()
agent = Agent(tools=DEMO_TOOLS, long_term_memory=ltm3)
print('Agent + 长期记忆已就绪')

### 自动记忆提取

用户说关于自己的事实时，LLM 自动调用 `save_memory`。

In [ ]:
reply = agent.chat('你好！我叫程宣赫，在人民大学读书，我喜欢 Python 和 AI。')
print(f'🤖 Agent: {reply}')

print('\n当前记忆:')
for m in ltm3.list_all():
    print(f"  [{m['index']}] {m['content']}")

### 跨会话回忆

新建 Agent（模拟新会话），传入同一个 LTM 实例。Agent 应该能"记起"之前的事。

In [ ]:
# 新建 Agent（模拟重启）
agent2 = Agent(tools=DEMO_TOOLS, long_term_memory=ltm3)

reply = agent2.chat('我叫什么名字？我在哪个学校？我喜欢什么？')
print(f'🤖 Agent: {reply}')

### 记忆 + 工具联动

Agent 用检索到的记忆来决定工具参数。

In [ ]:
# 存一个关于城市的事实
ltm3.add('用户住在北京，在海淀区')

agent3 = Agent(tools=DEMO_TOOLS, long_term_memory=ltm3)
reply = agent3.chat('我在的城市今天天气怎么样？')
print(f'🤖 Agent: {reply}')

# 清理
shutil.rmtree('agent_memory')
print('\n演示完成')

## 检索算法

当前使用**关键词匹配 + 时间衰减**：
1. 对 query 和每条记忆按字切分，计算交集/并集比例
2. 得分乘以时间衰减因子（半衰期约 30 天）
3. 按得分降序取 top_k

后续 05 工具发现实验会引入 embedding 检索，届时可以反哺优化这里的算法。

## 总结

| 组件 | 职责 |
|------|------|
| `LongTermMemory` | 持久化存储 + 关键词检索 |
| `save_memory` 工具 | LLM 主动调用，写入长期记忆 |
| `core.py` 增强 | 每轮 chat() 自动检索注入上下文 |
| CLI 命令 | `/remember` `/forget` `/memories` |

长期记忆让 Agent 从一个"失忆的对话机器人"变成了一个"会记住你"的助手。